# 11 - RAG Improvement

This notebook is about **how to make a weak RAG system better**.

The core principle is:

> Find the failing stage, change one thing, re-run the same benchmark, and keep the change only if it improves the useful metrics without unacceptable cost/latency.

This continues the evaluation ideas from `rag.ipynb`, while making the improvement loop explicit.

### Source foundation
This notebook is an expanded learning version of the uploaded notebooks. The source material for this track includes:
- `chunking.ipynb`
- `embeddings.ipynb`
- `faiss.ipynb`
- `chroma_db.ipynb`
- `searching.ipynb`
- `rag.ipynb`

The original examples and terminology are retained where useful, then organized into a cleaner learning sequence.

## 1. Failure taxonomy

| Symptom | Likely cause | First thing to test |
|---|---|---|
| Correct document never retrieved | chunking/embedding/query | inspect chunks and query representation |
| Correct doc retrieved but wrong chunk first | ranking problem | rerank / MMR / hybrid |
| Too many irrelevant chunks | precision problem | lower k / filter / rerank |
| Answer invents facts | grounding problem | prompt / model / faithfulness checks |
| Context too large | context problem | compression / lower k |
| System is slow | infrastructure/model problem | cache / fewer calls / smaller candidates |
| Cost too high | token/retrieval/model problem | reduce context / cache / route models |

In [ ]:
import pandas as pd

experiments = pd.DataFrame([
    {'name':'baseline','chunk_size':500,'overlap':50,'k':5,'rerank':False,'recall_at_5':0.68,'faithfulness':0.79,'latency_ms':900,'cost_proxy':1.0},
    {'name':'smaller_chunks','chunk_size':250,'overlap':50,'k':5,'rerank':False,'recall_at_5':0.76,'faithfulness':0.80,'latency_ms':920,'cost_proxy':1.0},
    {'name':'reranked','chunk_size':250,'overlap':50,'k':5,'rerank':True,'recall_at_5':0.76,'faithfulness':0.88,'latency_ms':1300,'cost_proxy':1.5},
])

display(experiments)

## 2. Improve chunking

Change chunk size and overlap only after you have a retrieval benchmark.

Good experiment:
```text
250/25
250/50
500/50
500/100
```

Measure Recall@K, MRR, and context size.

In [ ]:
def choose_best_by_metric(df, metric, higher_is_better=True):
    return df.sort_values(metric, ascending=not higher_is_better).iloc[0]

print(choose_best_by_metric(experiments, 'recall_at_5'))

## 3. Improve embeddings

Embedding changes can help when two texts have similar meaning but poor keyword overlap.

Test embeddings on the same corpus and question set. Re-index documents for each embedding model; mixing embedding spaces is not valid.

## 4. Improve retrieval

Try, in order appropriate to your failure:

1. top-k tuning
2. metadata filtering
3. MMR
4. keyword + semantic hybrid
5. query rewriting
6. multi-query
7. reranking

In [ ]:
def weighted_metric(row):
    # Example project-specific utility score. Tune weights from business goals.
    quality = 0.5 * row['recall_at_5'] + 0.5 * row['faithfulness']
    latency_penalty = max(0, row['latency_ms'] - 900) / 5000
    cost_penalty = max(0, row['cost_proxy'] - 1.0) / 4
    return quality - latency_penalty - cost_penalty

experiments['utility'] = experiments.apply(weighted_metric, axis=1)
print(experiments.sort_values('utility', ascending=False))

## 5. Improve the context

If retrieval finds the correct evidence but the LLM still struggles:

- remove duplicates
- rerank before generation
- compress long chunks
- preserve source labels
- put the most relevant evidence in a clear location
- avoid dumping the entire top-k into the prompt

## 6. Improve generation / grounding

A grounding prompt can say:

```text
Use only the supplied evidence.
Do not invent missing facts.
If evidence is insufficient, say so.
Cite the source of important claims.
```

Then measure whether faithfulness actually improves. Prompt changes should be validated, not celebrated because the answer sounds nicer.

## 7. Add a no-answer benchmark

A strong RAG system should not confidently answer questions that are outside the knowledge base.

Add examples where the correct behavior is:
```text
I don't know based on the provided documents.
```

## 8. Regression testing

Every change should be tested against:
- old successful questions
- old failure cases
- new edge cases
- out-of-scope questions

Keep a regression suite forever. Otherwise every improvement is just a future bug wearing a tie.

In [ ]:
regression_cases = [
    {'id':'q1','question':'What is RAG?','expected':'answerable'},
    {'id':'q2','question':'What is an embedding?','expected':'answerable'},
    {'id':'q3','question':'Who won a sports match not present in the corpus?','expected':'abstain'},
]

for case in regression_cases:
    print(case)

## 9. Improvement decision table

Before changing a component, write down the hypothesis:

```text
Problem: recall@5 is low.
Hypothesis: chunks are too small.
Change: 250 → 500 chars.
Expected: recall increases.
Risk: more irrelevant context.
Metric: recall@5 + context precision + latency.
```

This is experimental thinking applied to RAG.

## 10. Final improvement workflow

```text
baseline
  ↓
measure
  ↓
find weakest stage
  ↓
form hypothesis
  ↓
change ONE thing
  ↓
benchmark
  ↓
accept / reject
  ↓
repeat
```

## Final checklist

- [ ] I can classify a RAG failure.
- [ ] I can propose a targeted hypothesis.
- [ ] I can run a controlled experiment.
- [ ] I can compare quality, latency, and cost.
- [ ] I can build a regression suite.
- [ ] I know when to improve retrieval vs generation.

## Source / documentation notes

- The supplied `rag.ipynb` contains a basic evaluation set, retrieval testing, reranking, and citation-ready context.
- Ragas metric documentation: https://docs.ragas.io/en/latest/concepts/metrics/available_metrics/
- LangChain reference: https://reference.langchain.com/python/langchain/overview